In [1]:
import os

print(os.path.exists("house_data.csv"))

True


In [2]:
import pandas as pd

df = pd.read_csv("house_data.csv")

print(df.head())
print(df.shape)
print(df.columns)

   area_sqft  bedrooms  age_years  location_score     price
0        800         2         10               6   4500000
1       1000         2          8               7   5500000
2       1200         3          5               8   7000000
3       1500         3          4               9   9000000
4       1800         4          3               9  11000000
(20, 5)
Index(['area_sqft', 'bedrooms', 'age_years', 'location_score', 'price'], dtype='object')


In [3]:
print(df.columns.tolist())

['area_sqft', 'bedrooms', 'age_years', 'location_score', 'price']


In [5]:
X = df[
    ["area_sqft", "bedrooms", "age_years", "location_score"]
]

y = df["price"]

print("X:")
print(X.head())

print("\ny:")
print(y.head())

X:
   area_sqft  bedrooms  age_years  location_score
0        800         2         10               6
1       1000         2          8               7
2       1200         3          5               8
3       1500         3          4               9
4       1800         4          3               9

y:
0     4500000
1     5500000
2     7000000
3     9000000
4    11000000
Name: price, dtype: int64


In [6]:
import os

print("Current directory:", os.getcwd())
print("Model exists:", os.path.exists("model/model.joblib"))

Current directory: /home/sagemaker-user
Model exists: False


In [7]:
import os

print("Files in current directory:")
print(os.listdir("/home/sagemaker-user"))

Files in current directory:
['.bashrc', '.agent', '.kiro', '.claude', '.sagemaker_sql_editor_api_cache', '.local', '.ipython', '.npm', '.jupyter', 'untitled.chat', '.ipynb_checkpoints', '.cache', '.config', '.virtual_documents', 'day3.ipynb', 'train.py', 'house_data.csv', 'day3_source', 'Day04.ipynb', 'model.tar.gz', 'Day05.ipynb']


In [8]:
import tarfile
import os

archive = "/home/sagemaker-user/model.tar.gz"
extract_dir = "/home/sagemaker-user/day5_model"

os.makedirs(extract_dir, exist_ok=True)

with tarfile.open(archive, "r:gz") as tar:
    tar.extractall(extract_dir)

print("Extracted files:")
print(os.listdir(extract_dir))

Extracted files:
['inference.py', 'model.joblib']


/tmp/ipykernel_88825/1561890392.py:10: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(extract_dir)


In [9]:
import joblib

model_path = "/home/sagemaker-user/day5_model/model.joblib"

model = joblib.load(model_path)

print("Model loaded successfully!")
print(model)

Model loaded successfully!
RandomForestRegressor(random_state=42)


/opt/conda/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [11]:
with open("/home/sagemaker-user/train.py", "r") as f:
    print(f.read())

import os
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error


def main():

    # Local dataset
    # Get training data location from SageMaker
    input_dir = os.environ.get("SM_CHANNEL_TRAINING", ".")

    # Dataset path
    input_path = os.path.join(input_dir, "house_data.csv")

    # Read dataset
    df = pd.read_csv(input_path)

    # Features
    X = df[["area_sqft", "bedrooms", "age_years", "location_score"]]

    # Target
    y = df["price"]

    # Split data into training and testing
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    # Create Random Forest regression model
    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )

    # Train model
    model.fit(X_train, y_train)

    print("Model training completed!")

    # M

In [12]:
print("Model features:")
print(model.feature_names_in_)

Model features:
['area' 'bedrooms' 'bathrooms' 'age']


In [13]:
print("Current dataset features:")
print(X.columns.tolist())

Current dataset features:
['area_sqft', 'bedrooms', 'age_years', 'location_score']


In [17]:
import os

for root, dirs, files in os.walk("/home/sagemaker-user/day3_source"):
    for file in files:
        if file.endswith(".joblib") or file.endswith(".tar.gz"):
            print(os.path.join(root, file))

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model_correct = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model_correct.fit(X_train, y_train)

predictions = model_correct.predict(X_test)

mse = mean_squared_error(y_test, predictions)

print("Model trained successfully!")
print("Test MSE:", mse)

Model trained successfully!
Test MSE: 309401250000.0


In [19]:
import numpy as np

rmse = np.sqrt(mse)

print("MSE :", mse)
print("RMSE:", rmse)

MSE : 309401250000.0
RMSE: 556238.4830268398


In [20]:
results = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": predictions
})

print(results)

   Actual Price  Predicted Price
0       4500000        4720000.0
1      12000000       11010000.0
2       6500000        6856000.0
3       5500000        5787000.0


In [21]:
import os
import joblib

os.makedirs("/home/sagemaker-user/day5_artifact", exist_ok=True)

model_path = "/home/sagemaker-user/day5_artifact/model.joblib"

joblib.dump(model_correct, model_path)

print("Model artifact saved!")
print(model_path)

Model artifact saved!
/home/sagemaker-user/day5_artifact/model.joblib


In [22]:

import tarfile
import os

model_path = "/home/sagemaker-user/day5_artifact/model.joblib"
archive_path = "/home/sagemaker-user/day5_artifact/model.tar.gz"

with tarfile.open(archive_path, "w:gz") as tar:
    tar.add(model_path, arcname="model.joblib")

print("Model packaged successfully!")
print(archive_path)

Model packaged successfully!
/home/sagemaker-user/day5_artifact/model.tar.gz


In [23]:
import tarfile

archive_path = "/home/sagemaker-user/day5_artifact/model.tar.gz"

with tarfile.open(archive_path, "r:gz") as tar:
    print("Files inside model.tar.gz:")
    for name in tar.getnames():
        print(name)

Files inside model.tar.gz:
model.joblib


In [24]:
import boto3

s3 = boto3.client("s3", region_name="us-east-1")

response = s3.list_buckets()

print("Your S3 buckets:")
for bucket in response["Buckets"]:
    print(bucket["Name"])

Your S3 buckets:
day04
sagemaker-us-east-1-834176180969


In [25]:
import boto3

s3 = boto3.client("s3", region_name="us-east-1")

local_file = "/home/sagemaker-user/day5_artifact/model.tar.gz"
s3_key = "day5/model.tar.gz"

s3.upload_file(
    local_file,
    "day04",
    s3_key
)

print("Model uploaded successfully!")
print("S3 location: s3://day04/day5/model.tar.gz")

Model uploaded successfully!
S3 location: s3://day04/day5/model.tar.gz


In [26]:
import boto3

sagemaker_client = boto3.client(
    "sagemaker",
    region_name="us-east-1"
)

model_group_name = "Day5-HousePrice-Models"

response = sagemaker_client.create_model_package_group(
    ModelPackageGroupName=model_group_name,
    ModelPackageGroupDescription="Model versions for the Day 5 house price prediction model"
)

print("Model Group created!")
print("Name:", model_group_name)

Model Group created!
Name: Day5-HousePrice-Models


In [27]:
import boto3

sagemaker_client = boto3.client(
    "sagemaker",
    region_name="us-east-1"
)

model_package = sagemaker_client.create_model_package(
    ModelPackageGroupName="Day5-HousePrice-Models",
    ModelPackageDescription="Day 5 house price Random Forest model",
    InferenceSpecification={
        "Containers": [
            {
                "Image": "683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.4-2-py312-cpu-py3",
                "ModelDataUrl": "s3://day04/day5/model.tar.gz"
            }
        ],
        "SupportedContentTypes": [
            "application/json"
        ],
        "SupportedResponseMIMETypes": [
            "application/json"
        ]
    },
    ModelApprovalStatus="PendingManualApproval"
)

print("Model version registered!")
print("ARN:")
print(model_package["ModelPackageArn"])

Model version registered!
ARN:
arn:aws:sagemaker:us-east-1:834176180969:model-package/Day5-HousePrice-Models/1


In [29]:
print(response)

{'ModelPackageGroupName': 'Day5-HousePrice-Models', 'ModelPackageVersion': 1, 'ModelPackageRegistrationType': 'Registered', 'ModelPackageArn': 'arn:aws:sagemaker:us-east-1:834176180969:model-package/Day5-HousePrice-Models/1', 'ModelPackageDescription': 'Day 5 house price Random Forest model', 'CreationTime': datetime.datetime(2026, 9, 25, 12, 45, 31, 685000, tzinfo=tzlocal()), 'InferenceSpecification': {'Containers': [{'Image': '683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.4-2-py312-cpu-py3', 'ImageDigest': 'sha256:2ce4d4b8f312608308eeb9c9a89efb3a329eafcf399ea7c199bace80fceacee6', 'ModelDataUrl': 's3://day04/day5/model.tar.gz', 'ModelDataETag': '721998bb84d6918e56ca02248ea6b25f', 'IsCheckpoint': False}], 'SupportedContentTypes': ['application/json'], 'SupportedResponseMIMETypes': ['application/json']}, 'ModelPackageStatus': 'Completed', 'ModelPackageStatusDetails': {'ValidationStatuses': [], 'ImageScanStatuses': []}, 'CertifyForMarketplace': False, 'ModelApprov

In [30]:
print("Model Group:", response["ModelPackageGroupName"])
print("Version:", response["ModelPackageVersion"])
print("Status:", response["ModelPackageStatus"])
print("Approval:", response["ModelApprovalStatus"])
print("Artifact:", response["InferenceSpecification"]["Containers"][0]["ModelDataUrl"])

Model Group: Day5-HousePrice-Models
Version: 1
Status: Completed
Approval: PendingManualApproval
Artifact: s3://day04/day5/model.tar.gz


In [31]:
response = sagemaker_client.update_model_package(
    ModelPackageArn="arn:aws:sagemaker:us-east-1:834176180969:model-package/Day5-HousePrice-Models/1",
    ModelApprovalStatus="Approved"
)

print("Model approval updated!")
print("Approval status:", response["ModelApprovalStatus"])

Model approval updated!


KeyError: 'ModelApprovalStatus'

In [32]:
response = sagemaker_client.describe_model_package(
    ModelPackageName="arn:aws:sagemaker:us-east-1:834176180969:model-package/Day5-HousePrice-Models/1"
)

print("Model Group:", response["ModelPackageGroupName"])
print("Version:", response["ModelPackageVersion"])
print("Status:", response["ModelPackageStatus"])
print("Approval:", response["ModelApprovalStatus"])

Model Group: Day5-HousePrice-Models
Version: 1
Status: Completed
Approval: Approved
